In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. 설정
# ============================================================

SEED = 42
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 6
MAX_LEN = 128
BATCH_SIZE = 32

PROJECT_ROOT = Path.cwd()

TEST_PATH = PROJECT_ROOT / "results" / "splits" / "test.csv"

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "results"
    / "augmentation"
    / "wordnet_comparison"
    / "baseline"
    / "best_model.safetensors"
)

TOKENIZER_DIR = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "bert_tokenizer"
)

OUTPUT_DIR = PROJECT_ROOT / "results" / "final_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("Stage 12: Final Model Evaluation")
print("=" * 70)
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"Test data    : {TEST_PATH}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Output       : {OUTPUT_DIR}")


# ============================================================
# 2. 입력 파일 확인
# ============================================================

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"테스트 데이터 파일을 찾을 수 없습니다: {TEST_PATH}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "BERT 체크포인트를 찾을 수 없습니다.\n"
        f"현재 설정 경로: {CHECKPOINT_PATH}\n"
        "실제 best_model.safetensors 경로를 확인한 후 "
        "CHECKPOINT_PATH를 수정하세요."
    )


# ============================================================
# 3. 테스트 데이터 로드 및 기본 검증
# ============================================================

test_df = pd.read_csv(TEST_PATH)

required_columns = {"text", "label"}

if not required_columns.issubset(test_df.columns):
    raise ValueError(
        f"CSV에 필요한 컬럼이 없습니다. "
        f"필요한 컬럼: {required_columns}, "
        f"현재 컬럼: {list(test_df.columns)}"
    )

test_df = test_df[["text", "label"]].copy()

if test_df["text"].isna().any():
    raise ValueError("테스트 데이터에 결측 text가 있습니다.")

if test_df["label"].isna().any():
    raise ValueError("테스트 데이터에 결측 label이 있습니다.")

test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)

valid_labels = set(range(NUM_LABELS))
actual_labels = set(test_df["label"].unique())

if not actual_labels.issubset(valid_labels):
    raise ValueError(
        f"예상하지 못한 label이 있습니다: {actual_labels}"
    )

if actual_labels != valid_labels:
    print(
        "주의: 테스트 세트에 일부 클래스가 없을 수 있습니다.",
        f"발견된 클래스: {sorted(actual_labels)}"
    )

print("\n[1] Test dataset")
print(f"Number of samples: {len(test_df):,}")
print("\nClass distribution:")
print(
    test_df["label"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 4. 토크나이저 로드
# ============================================================

if TOKENIZER_DIR.exists():
    tokenizer_source = str(TOKENIZER_DIR)
else:
    tokenizer_source = MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)

print("\n[2] Tokenizer")
print(f"Tokenizer source: {tokenizer_source}")
print(f"Maximum length  : {MAX_LEN}")


# ============================================================
# 5. 모델 생성 및 최종 체크포인트 로드
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    attn_implementation="eager",
)

# safetensors 파일에 저장된 모델 가중치를 읽는다.
state_dict = load_file(
    str(CHECKPOINT_PATH),
    device="cpu",
)

# 전체 모델 가중치가 일치하는지 엄격하게 검사한다.
model.load_state_dict(state_dict, strict=True)

model.to(DEVICE)
model.eval()

print("\n[3] Model")
print(f"Model: {MODEL_NAME}")
print(f"Checkpoint loaded: {CHECKPOINT_PATH}")
print(
    f"Total parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ============================================================
# 6. 테스트 세트 추론
# ============================================================

all_logits = []
all_true_labels = []
all_texts = []

start_time = time.perf_counter()

with torch.inference_mode():

    for start_idx in range(0, len(test_df), BATCH_SIZE):

        batch_df = test_df.iloc[
            start_idx : start_idx + BATCH_SIZE
        ]

        texts = batch_df["text"].tolist()
        labels = batch_df["label"].to_numpy()

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        if USE_AMP:
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                outputs = model(**encoded)
        else:
            outputs = model(**encoded)

        logits = outputs.logits.float().cpu().numpy()

        all_logits.append(logits)
        all_true_labels.extend(labels.tolist())
        all_texts.extend(texts)

        completed = min(
            start_idx + BATCH_SIZE,
            len(test_df),
        )

        if (
            completed == len(test_df)
            or completed % 5000 < BATCH_SIZE
        ):
            print(
                f"\rInference progress: "
                f"{completed:,}/{len(test_df):,}",
                end="",
                flush=True,
            )

elapsed_seconds = time.perf_counter() - start_time

print("\nInference completed.")


# ============================================================
# 7. 예측값 및 확률 계산
# ============================================================

logits = np.concatenate(all_logits, axis=0)
y_true = np.asarray(all_true_labels)

# 수치적으로 안정적인 softmax 계산
shifted_logits = logits - logits.max(axis=1, keepdims=True)
exp_logits = np.exp(shifted_logits)

probabilities = exp_logits / exp_logits.sum(
    axis=1,
    keepdims=True,
)

y_pred = np.argmax(probabilities, axis=1)
confidence = probabilities.max(axis=1)

assert len(y_true) == len(y_pred) == len(test_df)


# ============================================================
# 8. 전체 평가 지표
# ============================================================

metrics = {
    "model": MODEL_NAME,
    "checkpoint": str(CHECKPOINT_PATH),
    "test_samples": int(len(y_true)),
    "max_length": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "accuracy": float(
        accuracy_score(y_true, y_pred)
    ),
    "macro_precision": float(
        precision_score(
            y_true,
            y_pred,
            labels=list(range(NUM_LABELS)),
            average="macro",
            zero_division=0,
        )
    ),
    "macro_recall": float(
        recall_score(
            y_true,
            y_pred,
            labels=list(range(NUM_LABELS)),
            average="macro",
            zero_division=0,
        )
    ),
    "macro_f1": float(
        f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_LABELS)),
            average="macro",
            zero_division=0,
        )
    ),
    "weighted_f1": float(
        f1_score(
            y_true,
            y_pred,
            labels=list(range(NUM_LABELS)),
            average="weighted",
            zero_division=0,
        )
    ),
    "mean_prediction_confidence": float(
        confidence.mean()
    ),
    "correct_predictions": int(
        (y_true == y_pred).sum()
    ),
    "incorrect_predictions": int(
        (y_true != y_pred).sum()
    ),
    "error_rate": float(
        (y_true != y_pred).mean()
    ),
    "inference_seconds": float(elapsed_seconds),
    "samples_per_second": float(
        len(test_df) / elapsed_seconds
    ),
}

with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metrics,
        f,
        ensure_ascii=False,
        indent=4,
    )

print("\n[4] Final test metrics")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# 9. 클래스별 평가 보고서
# ============================================================

class_names = [
    f"label_{i}"
    for i in range(NUM_LABELS)
]

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_LABELS)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).T

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv",
    encoding="utf-8-sig",
)

print("\n[5] Classification report")
print(
    report_df.round(4).to_string()
)


# ============================================================
# 10. 혼동 행렬 저장
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_LABELS)),
)

cm_df = pd.DataFrame(
    cm,
    index=[f"actual_{i}" for i in range(NUM_LABELS)],
    columns=[f"predicted_{i}" for i in range(NUM_LABELS)],
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig",
)

fig, ax = plt.subplots(figsize=(9, 7))

image = ax.imshow(cm, interpolation="nearest")

ax.set_title("BERT Final Test Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")

ax.set_xticks(range(NUM_LABELS))
ax.set_yticks(range(NUM_LABELS))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

threshold = cm.max() / 2

for row in range(NUM_LABELS):
    for col in range(NUM_LABELS):
        ax.text(
            col,
            row,
            f"{cm[row, col]:,}",
            ha="center",
            va="center",
            color=(
                "white"
                if cm[row, col] > threshold
                else "black"
            ),
        )

fig.colorbar(image, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)

plt.close(fig)


# ============================================================
# 11. 개별 예측 결과 저장
# ============================================================

prediction_df = pd.DataFrame({
    "text": all_texts,
    "true_label": y_true,
    "predicted_label": y_pred,
    "confidence": confidence,
    "is_correct": y_true == y_pred,
})

for label_idx in range(NUM_LABELS):
    prediction_df[f"prob_label_{label_idx}"] = (
        probabilities[:, label_idx]
    )

prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 12. 오분류 사례 저장
# ============================================================

errors_df = prediction_df[
    ~prediction_df["is_correct"]
].copy()

errors_df = errors_df.sort_values(
    "confidence",
    ascending=False,
)

errors_df.to_csv(
    OUTPUT_DIR / "test_misclassified_examples.csv",
    index=False,
    encoding="utf-8-sig",
)

# 실제 클래스 -> 예측 클래스별 오분류 빈도
pair_df = (
    errors_df
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="error_count")
    .sort_values("error_count", ascending=False)
)

pair_df.to_csv(
    OUTPUT_DIR / "test_confusion_pairs.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 13. 클래스별 Precision / Recall / F1 별도 저장
# ============================================================

per_class_df = report_df.loc[
    class_names,
    ["precision", "recall", "f1-score", "support"],
].copy()

per_class_df.index.name = "class"

per_class_df.to_csv(
    OUTPUT_DIR / "per_class_metrics.csv",
    encoding="utf-8-sig",
)


# ============================================================
# 14. 최종 결과 요약
# ============================================================

summary = {
    "selected_model": MODEL_NAME,
    "selection_basis": (
        "Validation Macro F1 and efficiency comparison"
    ),
    "test_evaluation_completed": True,
    "test_set_used_for_hyperparameter_tuning": False,
    "metrics": metrics,
    "output_files": [
        "final_metrics.json",
        "classification_report.csv",
        "per_class_metrics.csv",
        "confusion_matrix.csv",
        "confusion_matrix.png",
        "test_predictions.csv",
        "test_misclassified_examples.csv",
        "test_confusion_pairs.csv",
    ],
}

with open(
    OUTPUT_DIR / "final_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=4,
    )

print("\n" + "=" * 70)
print("Stage 12 completed.")
print(f"Results saved to: {OUTPUT_DIR}")
print("=" * 70)


Stage 12: Final Model Evaluation
Project root : /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis
Device       : cuda
Test data    : /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/splits/test.csv
Checkpoint   : /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/augmentation/wordnet_comparison/baseline/best_model.safetensors
Output       : /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/final_evaluation

[1] Test dataset
Number of samples: 65,355

Class distribution:
label
0    19036
1    22086
2     5425
3     8959
4     7491
5     2358

[2] Tokenizer
Tokenizer source: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/checkpoints/bert_tokenizer
Maximum length  : 128


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[3] Model
Model: bert-base-uncased
Checkpoint loaded: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/augmentation/wordnet_comparison/baseline/best_model.safetensors
Total parameters: 109,486,854
Inference progress: 65,355/65,355
Inference completed.

[4] Final test metrics
model: bert-base-uncased
checkpoint: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/augmentation/wordnet_comparison/baseline/best_model.safetensors
test_samples: 65355
max_length: 128
batch_size: 32
accuracy: 0.942805
macro_precision: 0.891983
macro_recall: 0.955856
macro_f1: 0.917724
weighted_f1: 0.944683
mean_prediction_confidence: 0.939047
correct_predictions: 61617
incorrect_predictions: 3738
error_rate: 0.057195
inference_seconds: 27.448040
samples_per_second: 2381.044336

[5] Classification report
              precision  recall  f1-score     support
label_0          0.9975  0.9573    0.9770  19036.0000
label_1          0.9983  0.9176    0.9563  22086.0000
labe